In [305]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import sys
import json
from datetime import datetime
import numpy as np


In [306]:
from typing import get_origin, get_args, Literal

In [307]:
MODEL = "qwen3:4b"

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",
)


In [308]:
class Memory:
    def __init__(self, embedding_function, file_path = "memory.json"):
        self.data = []
        self.next_id = 1
        self.embedding_function = embedding_function
        self.file_path = file_path

        self.load_from_disk()

    def save_to_disk(self):
        with open(self.file_path, "w", encoding="utf-8") as f:
            json.dump(self.data, f, indent=4)

    def load_from_disk(self):
        if not os.path.exists(self.file_path):
            return f"File path does not exist!!"

        with open(self.file_path, "r", encoding = "utf-8") as f:
            self.data = json.load(f)

        if self.data:
            self.next_id = max(memory["id"] for memory in self.data) + 1
            

    def remember(self, key, value, category = "other"):
        text = f"{key} : {value}"
        embedding = self.embedding_function(text)

        for memory in self.data:

            if memory["key"] == key:
                memory["value"] = value
                memory["text"] = text
                memory["embedding"] = embedding
                memory["category"] = category
                memory["timestamp"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                self.save_to_disk()

                return

        self.data.append({
            "id" : self.next_id,
            "key": key,
            "value": value,
            "text" : f"{key} : {value}",
            "embedding": embedding,
            "category": category,
            "timestamp" : datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        })
        
        self.next_id += 1
        self.save_to_disk()
    
    def recall(self, key):
        for memory in reversed(self.data):
            if memory["key"] == key:
                return memory["value"]

    def forget(self, key):
        original_count = len(self.data)

        self.data = [
            memory
            for memory in self.data
            if memory["key"] != key
        ]

        if len(self.data) < original_count:
            self.save_to_disk()
            return True

        return False

    def get_memories_by_categories(self, category):
        return [
            memory
            for memory in self.data
            if memory.get("category","other") == category
        ]

    def list_memories(self):
        return self.data

    def search(self, query, top_k = 3, threshold = 0.55, category = None):
        query_embedding = self.embedding_function(query)

        results = []

        for memory in self.data:

            if category is not None and memory.get("category") != category:
                continue

            score = cosine_similarity(query_embedding, memory["embedding"])

            if score >= threshold:
                results.append({
                    "memory" : memory,
                    "score" : float(round(score,3))
                })

        results.sort(
            key = lambda x : x["score"],
            reverse = True
        )

        return results[:top_k]


In [309]:
class Tool:
    def __init__(self, function, description):
        self.function = function
        self.description = description
        self.parameters = generate_parameters(function)

    def execute(self, arguments, context):
        try:
            if context is None:
                context = {}

            sig = inspect.signature(self.function)
            kwargs = dict(arguments)

            for k, v in context.items():
                if k in sig.parameters:
                    kwargs[k] = v
            
            return self.function(**kwargs)
        except Exception as e:
            return f"Tool execution failed: {str(e)}"

    def schema(self):
        return {
            "type" : "function",
            "function":{
                "name": self.function.__name__,
                "description": self.description,
                "parameters": self.parameters
            }
        }
    

In [310]:
class Agent:
    def __init__(self, client, tool_list, model, system_prompt):
        self.client = client
        self.model = model
        self.tool_list = tool_list

        self.TOOL_MAP = {
            tool.function.__name__ : tool
            for tool in tool_list
        }

        self.Tool_SCHEMA = [
            tool.schema()
            for tool in tool_list
        ]

        self.messages = [{
            "role": "system",
            "content": system_prompt
        }]

        self.state = {}
        self.memory = Memory(create_embedding, file_path = "memory.json")
        self.context = {"memory": self.memory}
        
    def set_state(self, key, value):
        self.state[key] = value

    def get_state(self, key):
        return self.state[key]

    def call_llm(self):
            return self.client.chat.completions.create(
                    model=self.model,
                    messages=self.messages,
                    tools=self.Tool_SCHEMA,
                    max_tokens=1000
                )

    def execute(self, tool_call):
            tool_name = tool_call.function.name
            print(f"TOOL CALLED: {tool_name}")
            tool = self.TOOL_MAP.get(tool_name)
            if not tool:
                return f"Tool '{tool_name}' does not exist."
            try:
                arguments = json.loads(tool_call.function.arguments)
                return tool.execute(arguments, self.context)
            except Exception as e:
                return f"Tool execution failed: {str(e)}"
        
    def run(self, user_input):
        self.messages.append({"role": "user", "content": user_input})
        MAX_ITERATIONS = 10
        for iteration in range(MAX_ITERATIONS):
            response = self.call_llm()
            response_message = response.choices[0].message
            self.messages.append(response_message)
            if not response_message.tool_calls:
                return f"AI: {response_message.content}"
            for tool_call in response_message.tool_calls:
                    print("Tool is running")
                    result = self.execute(tool_call)
                    self.messages.append({
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "name": tool_call.function.name,
                        "content": json.dumps(result)
                    })
        else:
            print("MAX Tool iterations reached!!")


In [311]:
EMBEDDING_MODEL = "Qwen3-Embedding:4B"

embedding_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",
)


In [312]:
def create_embedding(text):
    embedding = embedding_client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=text,
        encoding_format="float"
    )
    return embedding.data[0].embedding


In [313]:
def cosine_similarity(a, b):
    a = np.array(a)
    b = np.array(b)

    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [314]:
def python_type_to_json_type(annotation):

    if get_origin(annotation) is Literal:

        values = get_args(annotation)

        first_value = values[0]

        if isinstance(first_value, str):
            json_type = "string"
        
        elif isinstance(first_value, int):
            json_type = "integer"

        elif isinstance(first_value, float):
            json_type = "number"
        
        elif isinstance(first_value, bool):
            json_type = "boolean"

        else:
            json_type = "string"

        return{
            "type": json_type,
            "enum": list(values)
        }
    if annotation == str:
        return "string"

    elif annotation == int:
        return "integer"

    elif annotation == float:
        return "number"

    elif annotation == bool:
        return "boolean"

    return "string"

In [315]:
import inspect

def generate_parameters(function):
    
    signature = inspect.signature(function)

    properties = {}
    required = []

    for name, parameter in signature.parameters.items():
        if name == "memory":
            continue

        json_type = python_type_to_json_type(parameter.annotation)

        if isinstance(json_type, dict):
            properties[name] = json_type

        else:
            properties[name] = {
                "type" : json_type
            }

        if parameter.default is inspect.Parameter.empty:
            required.append(name)

    return {
        "type" : "object",
        "properties" : properties,
        "required" : required
    }

In [316]:
def get_current_time():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

In [317]:
time_tool = Tool(
    function = get_current_time,
    description="Get current Time",
)

In [318]:
def calculator(a: float, b: float, operation: Literal["add", "subtract", "multiply", "divide"]):
    if(operation == "add"):
        return a+b

    elif(operation == "divide"):
        if(b != 0):
            return a/b
        else: return "Cannot divide with Zero"

    elif(operation == "subtract"):
        return a-b
    
    elif(operation == "multiply"):
        return a*b


In [319]:
calculator_tool = Tool(
    function=calculator,
    description="Perform mathematical calculations",
)

In [320]:
def greet(name: str, age: int, excited: bool = False):
    if excited:
        return f"Hello {name}! You are {age} years old!"
    return f"Hello {name}. You are {age} years old."

In [321]:
greet_tool = Tool(
    greet,
    "Greets a Person"
)

In [322]:
def get_memories_by_category(memory: Memory, category: str = "other"):
    memories = memory.get_memories_by_categories(category)

    if not memories:
        return {
            "found" : False,
            "memories" : []
        }

    cleaned_memory = []

    for memory_data in memories:
        cleaned_memory.append({
            "key" : memory_data["key"],
            "value" : memory_data["value"],
            "category": memory_data.get("category","other")
        })

    return {
        "found" : True,
        "memories" : cleaned_memory
    }

In [323]:
memory_by_category_tool = Tool(
    get_memories_by_category,
    "For broad questions about what you know about the user, prefer retrieving memories by category or retrieving all stored memories rather than relying only on semantic search."
)

In [324]:
def save_memory(memory: Memory, key: str, value: str, category: str = "other"):
    memory.remember(key, value, category)
    return f"Remembered {key} = {value}"

In [325]:
memory_tool = Tool(
    save_memory,
    "Saves important information to agent's memory"
)

In [326]:
def recall_memory(memory: Memory, key: str):
    value = memory.recall(key)
    if value is None:
        return f"No memory found for '{key}'"

    return f"The value of {key} = {value}"

In [327]:
recall_tool = Tool(
    recall_memory,
    """Retrieve a memory using its exact key.

    Use this ONLY when you already know the exact memory key.
    For example, if the key is exactly "hometown", use:
    recall_memory(key="hometown").

    If you do not know the exact key, use search_memory instead."""
)

In [328]:
def forget_memory(memory: Memory, key: str):
    deleted = memory.forget(key)
    if not deleted:
        return f"No memory found : {key}"
    return f"Memory deleted : {key}"

In [329]:
forget_tool = Tool(
    forget_memory,
    """Forget (delete) a memory by its exact key.
    
    IMPORTANT: You must know the EXACT key before calling this.
    If unsure, call search_memory first to find the correct key,
    then call this with the exact key returned."""
)

In [330]:
def get_all_memories(memory: Memory):
    return memory.list_memories()

In [331]:
get_all_memories_tool = Tool(
    get_all_memories,
    "Used to get all the memories"
)

In [332]:
def search_memory(memory: Memory, query: str, category: str = None):
    print("Using search memory")
    results =  memory.search(query, category = category)

    if not results:
        return {
            "found": False,
            "memories": []
            }

    cleaned_results = []

    for result in results:
        memory_data = result["memory"]

        cleaned_results.append({
            "key": memory_data["key"],
            "value": memory_data["value"],
            "category": memory_data.get("category","other"),
            "score": float(round(result["score"], 3))
        })

    return {
        "found": True,
        "memories": cleaned_results
    }

In [333]:
search_memory_tool = Tool(
    search_memory,
    """Search the agent's memory using keywords when you are unsure of the
    exact memory key. Use this tool when the user asks about something
    that may be stored in memory but you do not know the exact key.

    Example:
    User asks "What programming language do I like?"
    Search using query="language".

    Do NOT use recall_memory unless you know the exact key."""
)

In [334]:
list_memory_tool = Tool(
    get_all_memories,
    "get all the memories currently stored by the agent"
)

In [335]:
import hashlib

In [336]:
def get_file_hash(file_path):
    sha256 = hashlib.sha256()

    with open(file_path, "rb") as f:
        while chunk := f.read(8192):
            sha256.update(chunk)

    return sha256.hexdigest()

In [337]:
from pypdf import PdfReader
from docx import Document

In [338]:
def load_document(file_path):

    extension = os.path.splitext(file_path)[1].lower()

    if extension == ".txt":

        with open(file_path, "r", encoding="utf-8") as f:
            return f.read()

    elif extension == ".pdf":

        reader = PdfReader(file_path)

        text = ""

        for page in reader.pages:
            page_text = page.extract_text()

            if page_text:
                text += page_text + "\n\n"

        return text

    elif extension == ".docx":

        document = Document(file_path)

        text = ""

        for paragraph in document.paragraphs:
            text += paragraph.text + "\n\n"

        return text

    else:
        raise ValueError(
            f"Unsupported file type: {extension}"
        )

In [339]:
def load_documents(folder_path):

    documents = []

    for filename in os.listdir(folder_path):

        file_path = os.path.join(folder_path, filename)

        if not os.path.isfile(file_path): 
            continue

        text = load_document(file_path)

        documents.append({
            "text" : text,
            "source" : file_path,
            "hash" : get_file_hash(file_path)
        })

    
    return documents

In [340]:
def chunk_text(text, source, chunk_size = 400, overlap = 50):
    chunks = []
    current_chunk = ""

    lines = [line.strip() for line in text.split("\n") if line.strip()]

    for line in lines:
        if len(current_chunk) + len(line) + 1 <= chunk_size:
            current_chunk += line + " "
        else:
            if current_chunk.strip():
                chunks.append(current_chunk.strip())

            while len(line) > chunk_size:
                chunks.append(line[:chunk_size].strip())
                line = line[chunk_size - overlap:]

            current_chunk = line + " "

    if current_chunk.strip():
        chunks.append(current_chunk.strip())

    file_hash = get_file_hash(file_path)
    return [
        {
            "text": chunk,
            "source": source,
            "chunk_id": i,
            "hash":file_hash
        }
        for i, chunk in enumerate(chunks)
    ]


In [341]:
def save_vector_store(chunk_embeddings, filepath = "vector_store.json"):
    with open(filepath, "w", encoding = "utf-8") as f:
        json.dump(chunk_embeddings, f)

In [342]:
def load_vector_store(filepath = "vector_store.json"):
    with open(filepath, "r", encoding = "utf-8") as f:
        return json.load(f)

In [343]:
existing_store = load_vector_store() if os.path.exists("vector_store.json") else []

cached_hashes = {item["source"]: item["hash"] for item in existing_store}

updated_embeddings = []

for document in load_documents("files"):
    file_path = document["source"]
    current_hash = document["hash"]

    if file_path in cached_hashes and cached_hashes[file_path] == current_hash:

        print(f"Skipping unchanged file: {file_path}")
        updated_embeddings.extend([c for c in existing_store if c["source"] == file_path])

    else:
        
        print(f"Embedding modified/new file: {file_path}")
        chunks = chunk_text(document["text"], file_path)
        for chunk in chunks:
            embedding = create_embedding(chunk["text"])
            updated_embeddings.append({
                "text": chunk["text"],
                "embedding": embedding,
                "source": chunk["source"],
                "chunk_id": chunk["chunk_id"],
                "hash": chunk["hash"]
            })

chunk_embeddings = updated_embeddings
save_vector_store(chunk_embeddings)


Embedding modified/new file: files\knowledge.txt
Embedding modified/new file: files\personal.txt
Embedding modified/new file: files\resume.pdf


In [344]:
for i, chunk in enumerate(chunks):
    print(f"\n--- Chunk {i} ---")
    print(chunk)


--- Chunk 0 ---
{'text': 'Devaguptapu V S Sai Ramesh ♂¶ap-¶arker-altVellore, Tamil Nadu, India ♂phone+91 79955 43536 ✉ sairameshdevaguptapu@gmail.com /githubSairamesh45 /linkedinsairamesh-devaguptapu /cer◎ifica◎eCertificates /codeLeetCode Education V ellore Institute of T echnologyVellore, Tamil Nadu B.Tech, Computer Science & Engineering (Blockchain Technology) Jul 2024 – May 2028', 'source': 'files\\resume.pdf', 'chunk_id': 0, 'hash': '07b77e7bc7ec2b1635f49008aa2c7ccaf6288bb918c24b4ba3933f155dac973e'}

--- Chunk 1 ---
{'text': '•CGPA: 9.51/10 Branch Rank: 3 Merit Award: 2024–25 & 2025–26 Achievvers Junior College 2024 Senior Secondary Education (Class XII); Percentage: 96.7% St. Ann’s EM High School 2022 Secondary School (Class X); Percentage: 94.66% Experience Backend Engineering InternJan 2026 – Jun 2026 MyPerro (Shark Tank India Featured Company) Hybrid', 'source': 'files\\resume.pdf', 'chunk_id': 1, 'hash': '07b77e7bc7ec2b1635f49008aa2c7ccaf6288bb918c24b4ba3933f155dac973e'}

---

In [345]:
def search_chunks(query, chunks, top_k = 2):
    results = []

    query_embedding = create_embedding(query)

    for chunk in chunks:
        score = cosine_similarity(
            query_embedding,
            chunk["embedding"]
        )

        results.append({
            "text" : chunk["text"],
            "score" : float(round(score,3)),
            "source" : chunk["source"],
            "chunk_id" : chunk["chunk_id"]
        })

    results.sort(
        key = lambda x : x["score"],
        reverse = True
    )

    return results[:top_k]
    

In [346]:
def search_documents(query):
    results = search_chunks(query, chunk_embeddings)

    results = [
        result
        for result in results
        if result["score"] >= 0.4
    ]

    return {
        "found": bool(results),
        "documents" : [
        {
            "text" : result["text"],
            "score" : round(result["score"],3),
            "source" : result["source"],
            "chunk_id" : result["chunk_id"]
        }
        for result in results
        ]
    }

In [347]:
search_documents_tool = Tool(
    search_documents,
    "Search the knowledge documents for relevant information."
)

In [348]:
tool_list = [
    calculator_tool,
    time_tool,
    greet_tool,
    memory_tool,
    recall_tool,
    forget_tool,
    search_memory_tool,
    memory_by_category_tool,
    get_all_memories_tool,
    search_documents_tool
]

In [349]:
def build_rag_prompt(query, results):
    context = "\n\n".join(
        result["text"]
        for result in results
    )

    return f"""
    Answer the user's question using ONLY the context below.

    If the answer cannot be found in the context, say that you don't know.

    CONTEXT:
    {context}

    QUESTION:
    {query}
    """

In [350]:
agent = Agent(
    client=client,
    tool_list=tool_list,
    model=MODEL,
    system_prompt="""You are a helpful AI agent.

You have access to tools for calculations, getting the current time,
and managing memory.

Use the calculator for mathematical calculations.
Use the time tool when the user asks for the current time.

Memory rules:

1. When the user explicitly asks you to remember something,
   use save_memory.

3. If you do NOT know the exact memory key, use search_memory.
   Do not guess the key.

4. For broad questions such as:
- "What do you know about me?"
- "What do you remember about me?"
- "Tell me what you know about me"

use get_all_memories instead of search_memory(when you cant use search memories by category)

5. Do not invent memories.

6. When the user asks you to forget something, ALWAYS use
   search_memory first to find the exact key, then call
   forget_memory with that exact key. Never guess the key.
   
7. When saving memories, classify them using these categories:

- identity: information that directly identifies the user, such as name, age, or occupation.
- preference: likes, dislikes, and favorites.
- personal: personal facts such as hometown, college, or background.
- context: temporary or current information such as current projects or goals.
- other: information that does not clearly fit the above categories.

8. When searching memory, use a category when the user's question clearly relates to a specific category.

Available categories:
- identity
- preference
- personal
- context
- other

If the question is broad or does not clearly belong to one category, search without specifying a category.

9. Memory update rules:

- Before saving a new personal fact, consider whether it modifies an existing memory.
- If the new information refers to an existing fact, reuse the existing key.
- Do not create duplicate keys for the same concept.
- If the user explicitly changes a preference or fact, update the existing memory.
- Do not store temporary conversational statements unless they are useful long-term.
- Store stable personal information such as identity, preferences, background, and ongoing projects.

10. For broad questions about what you know about the user, prefer retrieving memories by category or retrieving all stored memories rather than relying only on semantic search."""
)


In [351]:
while True:
    try:
        user_input = input("You: ")
    except (EOFError, KeyboardInterrupt):
        break

    if not user_input.strip():
        continue

    if user_input.lower().strip() == "exit":
        break

    try:
        response = agent.run(user_input)
        print(response)
    except Exception as e:
        print("Error:", e)